In [2]:
import os
import cv2
import dlib
import numpy as np
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import pickle
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
import collections
import math

In [3]:
detector = dlib.get_frontal_face_detector()
try:
    predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
except RuntimeError:
    print("❌ LỖI: Không tìm thấy file shape_predictor_68_face_landmarks.dat!")
    print("Vui lòng tải file http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 giải nén và để vào thư mục dự án.")
    sys.exit(1)

In [4]:
base_dataset_path = r"D:/hog_svm/dataset"
train_path = os.path.join(base_dataset_path, "train")
test_path = os.path.join(base_dataset_path, "test")
val_path = os.path.join(base_dataset_path, "val")

In [5]:
def extract_features(dataset_path, augment=False):
    X_features = [] 
    Y_labels = []   
    
    print(f"Đang trích xuất đặc trưng khuôn mặt từ {dataset_path}...")
    if not os.path.exists(dataset_path):
        print(f"Không tìm thấy thư mục: {dataset_path}")
        return X_features, Y_labels

    for person_name in os.listdir(dataset_path):
        person_dir = os.path.join(dataset_path, person_name)
        if not os.path.isdir(person_dir): continue
            
        for image_name in os.listdir(person_dir):
            image_path = os.path.join(person_dir, image_name)
            img = cv2.imread(image_path)
            if img is None: continue
                
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            # Chuyển tham số 1 thành 0 để không upsample ảnh, giúp quét cực kỳ nhanh
            faces = detector(gray, 0) # Detect mặt trong ảnh
            
            # Chỉ lấy ảnh nếu nó tìm thấy ĐÚNG 1 khuôn mặt để đảm bảo độ chính xác
            if len(faces) == 1:
                face = faces[0]
                
                # 1. Căn chỉnh khuôn mặt (Face Alignment) về 112x112 theo chuẩn tài liệu
                shape = predictor(img, face)
                face_chip = dlib.get_face_chip(img, shape, size=112)
                
                # 2. Chuyển về ảnh xám và resize về 128x128 để đưa vào HOG
                gray_chip = cv2.cvtColor(face_chip, cv2.COLOR_BGR2GRAY)
                face_img_resized = cv2.resize(gray_chip, (128, 128))
                
                # Tính toán vector HOG của khuôn mặt (vector dài 1764)
                hog_feature = hog(
                    face_img_resized, 
                    orientations=9, 
                    pixels_per_cell=(8, 8), 
                    cells_per_block=(2, 2), 
                    block_norm='L2-Hys', 
                    visualize=False)
                
                X_features.append(hog_feature)
                Y_labels.append(person_name)
                
                # Áp dụng Data Augmentation (Lật ngang ảnh) để tăng gấp đôi dữ liệu train
                if augment:
                    flipped_img = cv2.flip(face_img_resized, 1)
                    hog_flipped = hog(flipped_img, orientations=9, pixels_per_cell=(8, 8), 
                                      cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
                    X_features.append(hog_flipped)
                    Y_labels.append(person_name)
    return X_features, Y_labels

# 1. Trích xuất đặc trưng và Train mô hình (có bật Augmentation)
X_train, Y_train = extract_features(train_path, augment=True)
print(f"Đã trích xuất được {len(X_train)} khuôn mặt từ tập train (đã bao gồm ảnh lật ngang).")

Đang trích xuất đặc trưng khuôn mặt từ D:/hog_svm/dataset\train...
Đã trích xuất được 602 khuôn mặt từ tập train (đã bao gồm ảnh lật ngang).


In [6]:
le = LabelEncoder()
Y_train_encoded = le.fit_transform(Y_train)

In [ ]:
print("Đang chạy GridSearchCV để tìm siêu tham số tối ưu nhất...")
param_grid = {
    'C': [1, 10, 100],
    'gamma': [1e-3, 5e-4, 1e-4, 'scale'],
    'kernel': ['rbf', 'linear']
}
base_svm = SVC(class_weight='balanced', probability=True)
svm_model = GridSearchCV(base_svm, param_grid, cv=5, n_jobs=-1, verbose=1)
svm_model.fit(X_train, Y_train_encoded)

print(f"⭐ Tham số tốt nhất tìm được: {svm_model.best_params_}")

Đang chạy GridSearchCV để tìm siêu tham số tối ưu nhất...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


In [ ]:
%matplotlib inline
matplotlib.use('module://ipykernel.pylab.backend_inline')

# ====================================================
# ⚙️ NGƯỠNG NHẬN DIỆN - chỉnh tại đây
# ====================================================
CONFIDENCE_THRESHOLD = 0.6

def predict_with_threshold(X, threshold=CONFIDENCE_THRESHOLD):
    """Dự đoán có ngưỡng: dưới ngưỡng → trả về 'Unknown'."""
    probs = svm_model.predict_proba(X)
    max_probs = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    labels = []
    for prob, pred in zip(max_probs, predictions):
        if prob >= threshold:
            labels.append(le.inverse_transform([pred])[0])
        else:
            labels.append("Unknown")
    return labels

def print_classification_report(Y_true, Y_pred, classes, dataset_name):
    print(f"\n📊 Bảng Classification Report ({dataset_name}) - ngưỡng={CONFIDENCE_THRESHOLD}):")
    all_classes = list(classes) + ["Unknown"]
    print(classification_report(Y_true, Y_pred, labels=all_classes, zero_division=0))

def draw_confusion_matrix(Y_true, Y_pred, classes, dataset_name, ax):
    all_classes = list(classes) + ["Unknown"]
    cm = confusion_matrix(Y_true, Y_pred, labels=all_classes)
    cm_df = pd.DataFrame(cm, index=all_classes, columns=all_classes)
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix - {dataset_name} (ngưỡng={CONFIDENCE_THRESHOLD})')
    ax.set_ylabel('Thực tế')
    ax.set_xlabel('Dự đoán')

X_test, Y_test = extract_features(test_path)
Y_pred_test = predict_with_threshold(X_test)
print_classification_report(Y_test, Y_pred_test, le.classes_, "Test")

X_val, Y_val = extract_features(val_path)
Y_pred_val = predict_with_threshold(X_val)
print_classification_report(Y_val, Y_pred_val, le.classes_, "Validation")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
draw_confusion_matrix(Y_test, Y_pred_test, le.classes_, "Test", axes[0])
draw_confusion_matrix(Y_val, Y_pred_val, le.classes_, "Validation", axes[1])
plt.tight_layout()
plt.show()

NameError: name 'matplotlib' is not defined

In [ ]:
with open("models_svm.pkl", "wb") as f:
    pickle.dump((le, svm_model, CONFIDENCE_THRESHOLD), f)

print(f"✅ Đã lưu mô hình vào models_svm.pkl (ngưỡng={CONFIDENCE_THRESHOLD})")

✅ Đã học xong mô hình và lưu vào file models_svm.pkl!
